In [1]:
import os, uuid, hashlib, platform, getpass
# ===== 1. 填写学号 =====
STUDENT_ID = 20244080230
# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] exp01
[学号]     20244080230
[机器指纹] 2d80c6ac8fb85281
[RUN_ID]   20244080230-2d80c6ac8fb85281


## 一、热身：定义简单函数

- `say_hi()`：不接受参数，直接打印固定字符串。
- `goat_pad(s)`：用字符串相加 `"goat" + s + "goat"`，在前后各加 `goat`。
- `print_n(s, n)`：用 `for` 循环配合 `range(n)` 把 `s` 打印 `n` 次；`n = 0` 时循环体一次都不执行，
  正好是"打印 0 次"。

三个函数的要求都是"打印"，所以用 `print()`，不需要 `return`。

In [2]:
def say_hi():
    print("Hello, world!")


def goat_pad(s):
    print("goat" + s + "goat")


def print_n(s, n):
    for _ in range(n):
        print(s)


say_hi()
goat_pad("Hello, world!")
print_n("Python", 3)

Hello, world!
goatHello, world!goat
Python
Python
Python


## 二、欧几里得算法

利用恒等式 $\gcd(m,\ n) = \gcd(n,\ m \bmod n)$：反复做带余除法，余数严格变小，最终余数为 0 时
除数就是最大公约数。对应代码 `while n != 0: m, n = n, m % n`，循环结束返回 `m`。
`n = 0` 时循环不执行，直接返回 `m`，正好符合 `gcd(m, 0) = m`。

In [3]:
def gcd(m, n):
    while n != 0:
        m, n = n, m % n
    return m


for m, n in [(2018, 2019), (1200, 300), (5040, 60)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


### 参数是负数时会怎样？这种行为了有意义吗？

Python 的 `%` 结果**跟随除数的符号**（`-2018 % 2019 == 1`，而 `2018 % -2019 == -1`），所以函数不会报错，
但返回的是**带符号的结果**：`n != 0` 时符号与第二个参数 `n` 相同；`n == 0` 时直接返回 `m`
（因此 `gcd(-5, 0)` 得到 `-5`）。只有 `abs(gcd(m, n))` 才是真正的 GCD。

这样的行为**没有数学意义**：最大公约数按惯例定义为非负整数——它是公共因子集合中的最大值，
不可能为负，标准库 `math.gcd(-8, 12) == 4` 也是如此。负号只是实现细节的副作用，还会破坏对称性：
`gcd(2018, -2019)` 得到 -1，而交换参数后 `gcd(-2019, 2018)` 得到 1。

修正办法是在进入循环前对两个参数取 `abs`。

In [4]:
print(f"{'m':>8}{'n':>8}{'gcd(m, n)':>12}{'abs(gcd)':>10}")
for m, n in [(-2018, 2019), (2018, -2019), (-8, 12), (8, -12), (-5, 0), (0, -7)]:
    r = gcd(m, n)
    print(f"{m:>8}{n:>8}{r:>12}{abs(r):>10}")

       m       n   gcd(m, n)  abs(gcd)
   -2018    2019           1         1
    2018   -2019          -1         1
      -8      12           4         4
       8     -12          -4         4
      -5       0          -5         5
       0      -7          -7         7


## 三、近似欧拉数 e

真实值 `e = math.exp(1) ≈ 2.718281828459045`。

1. `euler_limit(n)`：按公式 (2) 返回 `(1 + 1/n) ** n`。`1 / n` 是浮点除法，所以结果是 `float`。
   该数列收敛很慢（误差约正比于 1/(2n)）。
2. `euler_infinite_sum(n)`：公式 (1) 前 n 项之和，即 k = 0, 1, …, n−1 的 1/k! 之和
   （函数按 1-indexed 计数）。用滚动阶乘避免重复计算，复杂度 O(n)。
   按规定 `euler_infinite_sum(0) = 0`、`(1) = 1`、`(4) = 1 + 1 + 1/2 + 1/6 ≈ 2.6667`。
3. `euler_approx(eps)`：用 `while` 循环累加公式 (1) 的项，直到 `abs(部分和 − e) < eps` 就返回。
   部分和从下方单调逼近 e，收敛很快；受双精度浮点限制，最小误差约 4.4e-16。
4. `print_euler_sum_table(n)`：对 k = 1..n 逐行打印 `euler_infinite_sum(k)`。

In [5]:
import math


def euler_limit(n):
    return (1 + 1 / n) ** n


print("真实值 math.exp(1) =", math.exp(1))
print(f"{'n':>12}{'euler_limit(n)':>22}{'|误差|':>14}")
for n in [2, 10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    approx = euler_limit(n)
    print(f"{n:>12}{approx:>22.15f}{abs(approx - math.exp(1)):>14.3e}")

真实值 math.exp(1) = 2.718281828459045
           n        euler_limit(n)          |误差|
           2     2.250000000000000     4.683e-01
          10     2.593742460100002     1.245e-01
         100     2.704813829421528     1.347e-02
        1000     2.716923932235594     1.358e-03
       10000     2.718145926824926     1.359e-04
      100000     2.718268237192297     1.359e-05
     1000000     2.718280469095753     1.359e-06


In [6]:
def euler_infinite_sum(n):
    total = 0.0
    factorial = 1          # 0! = 1
    for k in range(n):
        if k > 0:
            factorial *= k  # 由 (k-1)! 滚动得到 k!
        total += 1.0 / factorial
    return total


print("euler_infinite_sum(0) =", euler_infinite_sum(0))
print("euler_infinite_sum(1) =", euler_infinite_sum(1))
print("euler_infinite_sum(4) =", euler_infinite_sum(4))

euler_infinite_sum(0) = 0.0
euler_infinite_sum(1) = 1.0
euler_infinite_sum(4) = 2.6666666666666665


In [7]:
def euler_approx(eps):
    true_e = math.exp(1)
    total = 0.0
    factorial = 1.0
    k = 0
    while abs(total - true_e) >= eps:
        if k > 0:
            factorial *= k
        total += 1.0 / factorial
        k += 1
    return total


for eps in [1e-1, 1e-3, 1e-6, 1e-10, 1e-15]:
    approx = euler_approx(eps)
    print(f"eps = {eps:.0e} → {approx:.15f}   误差 = {abs(approx - math.exp(1)):.3e}")

eps = 1e-01 → 2.666666666666667   误差 = 5.162e-02
eps = 1e-03 → 2.718055555555555   误差 = 2.263e-04
eps = 1e-06 → 2.718281525573192   误差 = 3.029e-07
eps = 1e-10 → 2.718281828446759   误差 = 1.229e-11
eps = 1e-15 → 2.718281828459046   误差 = 4.441e-16


In [8]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print("print_euler_sum_table(5)：")
print_euler_sum_table(5)

print_euler_sum_table(5)：
1.0
2.0
2.5
2.6666666666666665
2.708333333333333


## 四、测试整数属性

只用整除 `//` 和取模 `%`，不用 `math.sqrt`：如果 n 是 b 的幂，用 b 反复整除 n 一定能恰好除到 1；
中途除不尽就说明剩下的数里含有 b 以外的因子。

循环条件写成 `n % b == 0 and n > 1`：前半句保证当前数还能被 b 整除，后半句用来在 n 变成 1 时停住
（否则 `1 // b == 0` 会继续算下去）。题目约定 p ≥ 0，所以 `1 = b^0` 也应返回 `True`。

`is_power_of_2` 就是 `is_power` 在 b = 2 时的写法，是同一个思路的推广。另外 `b = 1` 时
`n % 1 == 0` 恒成立、`n // 1` 也等于 n，循环会永不结束，所以显式处理这个退化情形。

In [9]:
def is_power_of_2(n):
    while n % 2 == 0 and n > 1:
        n //= 2
    return n == 1


for n in [1, 2, 3, 4, 6, 8, 16, 1023, 1024, 4096]:
    print(f"is_power_of_2({n:>5}) = {is_power_of_2(n)}")

is_power_of_2(    1) = True
is_power_of_2(    2) = True
is_power_of_2(    3) = False
is_power_of_2(    4) = True
is_power_of_2(    6) = False
is_power_of_2(    8) = True
is_power_of_2(   16) = True
is_power_of_2( 1023) = False
is_power_of_2( 1024) = True
is_power_of_2( 4096) = True


In [10]:
def is_power(b, n):
    if b == 1:              # 1 的幂只有 1 本身，否则 n % 1 == 0 会死循环
        return n == 1
    while n % b == 0 and n > 1:
        n //= b
    return n == 1


for b, n in [(2, 8), (2, 12), (3, 27), (3, 10), (4, 64), (4, 8), (5, 125), (10, 1000), (1, 5)]:
    print(f"is_power({b}, {n}) = {is_power(b, n)}")

is_power(2, 8) = True
is_power(2, 12) = False
is_power(3, 27) = True
is_power(3, 10) = False
is_power(4, 64) = True
is_power(4, 8) = False
is_power(5, 125) = True
is_power(10, 1000) = True
is_power(1, 5) = False
